In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv('augmented_data.csv', index_col=0)
df

,Moisture_Content,Tablet_Weight,Hardness,Friability,Disintegration_Time,Dissolution_Rate,Content_Uniformity,Time_Minutes,Temperature_C,Pressure_Bar,Humidity_Percent,Motor_Speed_RPM,Compression_Force_kN,Flow_Rate_LPM,Power_Consumption_kW,Vibration_mm_s,Compression_Load,Thermal_Energy,Mechanical_Energy,Speed_Flow
0,2.1,205.065634,43.403346,1.498755,13.410275,68.600450,92.312002,-16.510038,53.610489,1.085227,39.923782,-97.840711,-0.725621,6.101625,2.555416,-0.797990,-0.787464,-885.111202,70.995300,-596.987368
1,2.1,197.732948,75.775032,0.773016,14.295473,35.317008,93.692700,136.054911,26.049085,1.044876,23.213324,65.059809,10.448520,-0.905872,12.822853,10.865501,10.917406,3544.105969,679.778718,-58.935861
2,2.1,208.047506,38.732884,1.661347,13.481166,72.955167,92.930091,-41.018417,57.767625,0.801366,47.415794,129.626057,-5.338327,6.787180,-2.612494,-4.129345,-4.277952,-2369.536554,-691.986220,879.795353
3,2.1,202.547384,73.278319,1.000286,13.841320,40.591516,91.356841,100.035336,34.551797,0.681428,36.367410,165.555576,8.054173,2.048184,12.401410,8.483883,5.488338,3456.400625,1333.413236,339.088244
4,2.1,202.063983,62.716305,1.334016,14.011245,48.548385,92.484603,50.333785,38.338109,0.836350,47.022921,294.395325,2.362261,0.611096,4.404706,4.432846,1.975676,1929.702125,695.438635,179.903905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2.1,208.492372,61.249880,1.144512,14.129530,52.291352,92.336442,89.660175,44.899090,0.927187,34.308171,578.883675,-1.496121,2.760840,2.891193,2.306861,-1.387185,4025.660277,-866.080190,1598.205302
9996,2.1,195.533921,64.178406,1.262149,14.693474,39.000015,92.445340,55.437715,11.591003,1.152354,39.933994,162.679986,4.438120,-3.886563,5.880318,5.367409,5.114286,642.578739,721.993366,-632.266018
9997,2.1,199.717364,72.978663,0.936103,14.718757,37.970363,93.295279,242.930056,26.749129,0.893230,35.551755,47.146600,9.703695,0.433486,12.753587,9.552231,8.667633,6498.167436,457.496231,20.437404
9998,2.1,199.798371,37.426292,1.780250,13.311269,64.853988,92.117517,40.890254,34.827292,1.259852,55.728258,-96.433692,-3.119364,1.962144,-2.093132,-2.720024,-3.929938,1424.096809,300.811812,-189.216769


In [7]:
df["Temp_Pressure"] = df["Temperature_C"] * df["Pressure_Bar"]

df["Humidity_Temp"] = df["Humidity_Percent"] * df["Temperature_C"]

df["Time_Temp"] = df["Time_Minutes"] * df["Temperature_C"]

df["Speed_Pressure"] = df["Motor_Speed_RPM"] * df["Pressure_Bar"]

df["Speed_Time"] = df["Motor_Speed_RPM"] * df["Time_Minutes"]

df.columns

Index(['Moisture_Content', 'Tablet_Weight', 'Hardness', 'Friability',
       'Disintegration_Time', 'Dissolution_Rate', 'Content_Uniformity',
       'Time_Minutes', 'Temperature_C', 'Pressure_Bar', 'Humidity_Percent',
       'Motor_Speed_RPM', 'Compression_Force_kN', 'Flow_Rate_LPM',
       'Power_Consumption_kW', 'Vibration_mm_s', 'Compression_Load',
       'Thermal_Energy', 'Mechanical_Energy', 'Speed_Flow', 'Temp_Pressure',
       'Humidity_Temp', 'Time_Temp', 'Speed_Pressure', 'Speed_Time'],
      dtype='str')

In [9]:
X = df.drop(['Hardness','Content_Uniformity','Dissolution_Rate','Tablet_Weight','Friability','Disintegration_Time'],axis=1)
y = df[['Hardness','Content_Uniformity','Dissolution_Rate','Tablet_Weight','Friability','Disintegration_Time']]

In [12]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.1,random_state=42)
X_train.shape

(9000, 19)

In [13]:
#Handling Outliers (Robust Scaling)
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

models = {

    "RandomForest": MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=800,
            max_depth=20,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            bootstrap=True,
            n_jobs=-1,
            random_state=42
        )
    ),

    "XGBoost": MultiOutputRegressor(
        XGBRegressor(
            n_estimators=2000,
            learning_rate=0.02,
            max_depth=8,
            min_child_weight=3,
            subsample=0.9,
            colsample_bytree=0.8,
            gamma=0.1,
            reg_alpha=0.1,
            reg_lambda=2,
            objective="reg:squarederror",
            n_jobs=-1,
            random_state=42
        )
    ),

    "GradientBoosting": MultiOutputRegressor(
        GradientBoostingRegressor(
            n_estimators=800,
            learning_rate=0.03,
            max_depth=5,
            min_samples_split=5,
            min_samples_leaf=2,
            subsample=0.9,
            random_state=42
        )
    ),

    "SVR": MultiOutputRegressor(
        SVR(
            kernel="rbf",
            C=50,
            epsilon=0.05,
            gamma="scale"
        )
    )
}

In [15]:

# Training + Evaluation

from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

results = []

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train_scaled, y_train)

    preds = model.predict(X_test_scaled)

    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    results.append({
        "Model": name,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse
    })

Training RandomForest...
Training XGBoost...
Training GradientBoosting...
Training SVR...


In [16]:
results_df = pd.DataFrame(results).sort_values(by="R2", ascending=False)

print("\nModel Performance")
print(results_df)


Model Performance
              Model        R2       MAE      RMSE
2  GradientBoosting  0.815532  0.407030  0.575036
1           XGBoost  0.812033  0.389928  0.562116
0      RandomForest  0.810924  0.423068  0.642911
3               SVR  0.762096  0.379509  0.706232
